# AI for NTH Summer School — Environment Verification

Please run the cells in order. This notebook checks that the teaching environment is working correctly before the summer school practical sessions begin.

It verifies:
- Python version
- required package imports
- simple numerical and plotting functionality
- basic scikit-learn models
- openpyxl write test
- PyTorch CPU forward pass
- expected bundle folders

**Expected result:** all checks should complete without errors.

## 1. Python and platform information

In [ ]:
import sys, platform
from pathlib import Path

print('Python executable:', sys.executable)
print('Python version   :', platform.python_version())
print('Platform         :', platform.platform())
print('Machine          :', platform.machine())
print('Working directory:', Path.cwd())

assert sys.version_info >= (3, 11), 'Python 3.11 or newer is required.'
print('\nPASS: Python version is suitable.')

## 2. Required imports

In [ ]:
import importlib

required = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'matplotlib': 'matplotlib',
    'scikit-learn': 'sklearn',
    'joblib': 'joblib',
    'openpyxl': 'openpyxl',
    'pyyaml': 'yaml',
    'torch': 'torch',
    'jupyterlab': 'jupyterlab',
}

mods = {}
for label, module_name in required.items():
    mod = importlib.import_module(module_name)
    mods[module_name] = mod
    print(f"PASS: {label:12s} -> {getattr(mod, '__version__', 'unknown')}")

## 3. Bundle folder check

This cell tries to locate the summer school bundle root and check that the expected folders exist.

In [ ]:
from pathlib import Path

here = Path.cwd().resolve()
candidates = [
    here,
    here.parent,
    here.parent.parent,
]

bundle_root = None
for cand in candidates:
    if (cand / 'course_materials').exists() or (cand / 'datasets').exists() or (cand / 'environment').exists():
        bundle_root = cand
        break

if bundle_root is None:
    bundle_root = here

print('Bundle root guess:', bundle_root)
for name in ['course_materials', 'datasets', 'environment']:
    path = bundle_root / name
    print(f"{'PASS' if path.exists() else 'WARN'}: {path}")


## 4. NumPy and pandas smoke test

In [ ]:
import numpy as np
import pandas as pd

arr = np.array([[1.0, 2.0], [3.0, 4.0]])
df = pd.DataFrame(arr, columns=['a', 'b'])
display(df)
assert abs(df['a'].mean() - 2.0) < 1e-12
print('PASS: NumPy and pandas basic operations are working.')

## 5. Matplotlib plot test

In [ ]:
import matplotlib.pyplot as plt

x = [0, 1, 2]
y = [0, 1, 4]
plt.figure(figsize=(5, 3))
plt.plot(x, y, marker='o')
plt.title('Environment verification plot')
plt.xlabel('x')
plt.ylabel('y')
plt.tight_layout()
plt.show()
print('PASS: Matplotlib plotting works.')

## 6. scikit-learn smoke tests

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

X = np.linspace(0.0, 1.0, 20).reshape(-1, 1)
y = (X[:, 0] ** 2 + 0.1).astype(float)

lin = LinearRegression().fit(X, y)
lin_pred = lin.predict([[0.5]])[0]

rf = RandomForestRegressor(n_estimators=10, max_depth=3, random_state=0, n_jobs=1).fit(X, y)
rf_pred = rf.predict([[0.5]])[0]

kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-4)
gp = GaussianProcessRegressor(kernel=kernel, random_state=0).fit(X, y)
gp_mean, gp_std = gp.predict([[0.5]], return_std=True)

print(f'PASS: LinearRegression prediction at x=0.5 -> {lin_pred:.6f}')
print(f'PASS: RandomForest prediction at x=0.5    -> {rf_pred:.6f}')
print(f'PASS: GP prediction mean/std at x=0.5     -> {gp_mean[0]:.6f}, {gp_std[0]:.6f}')

## 7. openpyxl write test

In [ ]:
from openpyxl import Workbook
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    out = Path(tmpdir) / 'openpyxl_test.xlsx'
    wb = Workbook()
    ws = wb.active
    ws['A1'] = 'AI for NTH'
    ws['A2'] = 42
    wb.save(out)
    assert out.exists() and out.stat().st_size > 0

print('PASS: openpyxl workbook creation and save worked.')

## 8. PyTorch CPU smoke test

In [ ]:
import torch
from torch import nn

x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
model = nn.Sequential(nn.Linear(2, 4), nn.ReLU(), nn.Linear(4, 1))
with torch.no_grad():
    y = model(x)

print('PyTorch version:', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
print('Output shape   :', tuple(y.shape))
assert tuple(y.shape) == (2, 1)
print('PASS: PyTorch CPU test succeeded.')

## 9. JupyterLab availability

In [ ]:
import subprocess

completed = subprocess.run(
    [sys.executable, '-m', 'jupyter', 'lab', '--version'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    check=True,
)
version = completed.stdout.strip() or completed.stderr.strip() or 'unknown'
print('JupyterLab version:', version)
print('PASS: JupyterLab is available from the active environment.')

## 10. Final status

If all cells above have run successfully, the environment is ready for the summer school practical sessions.

**Next step:** move to the Day 1 materials.